# 01 — TRIBE verification (decision gate 17)

Track A, Colab GPU. Verifies the pinned checkpoint against
`config/checkpoints.lock`, reproduces Meta's published example, records the
environment, and writes `outputs/verification/gate17.json`.

**`scripts/run_tribe_inference.py` refuses to touch a project stimulus until
that artifact exists for the pinned revision.** Gate 17 is enforced in code,
not on a checklist. It is a *project* gate, separate from HuggingFace access:
having Llama-3.2-3B approved is what makes this notebook runnable, not what
makes it unnecessary.

Nothing below computes anything itself — every cell is a bootstrap step or a
call into `scripts/`.

## Bootstrap

1. **GPU check** — Track A needs one. Runtime → Change runtime type → GPU.
2. **Clone `phase2-tribe` and install**, then **Runtime → Restart session**.
   The branch is not optional: the repo's default branch has no `src/tribe/`.
3. **Session setup** — mount Drive, derive the cache paths, set `HF_HOME`.
4. **Authenticate** from Colab Secrets (🔑), never from a literal in a cell.

Steps 3 and 4 re-derive everything they need, so they are what you re-run after
a restart or a dropped connection — nothing above them is needed again.

`HF_HOME` must be set *before* any HuggingFace import in the process. Once a HF
module is imported the cache location is fixed, and several GB of checkpoint
land on the ephemeral runtime disk instead of Drive.

Everything after that is a single call into a script in `scripts/`. No project
logic lives in this notebook: a cell dies with the session, and brief §4.3
requires every result to come from a script in the repo.

In [ ]:
# 1. GPU check
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")

In [ ]:
# 2. Clone the repo and install the pinned Track A stack into Colab's interpreter
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/MatteoGuardamagna4/neurotutorsim.git"
# Not optional. Phase II lives on `phase2-tribe`; the default branch (`master`)
# has no src/tribe/ and no scripts/run_tribe_*.py, so an unpinned clone fails
# further down with a confusing "No such file or directory".
BRANCH = "phase2-tribe"
REPO_DIR = Path("/content/neurotutorsim")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
# --system installs into Colab's own interpreter. `uv sync` would build a venv
# this kernel cannot import from.
#
# `.[tribe,dev]`, NOT `--extra tribe --extra dev`: in pip mode uv rejects --extra
# unless the target is `-r <file>`. With `-e .` it exits 2 with
#   "Requesting extras requires a ... pyproject.toml ... Use <dir>[extra] instead"
subprocess.run(
    ["uv", "pip", "install", "--system", "-e", ".[tribe,dev]"],
    cwd=REPO_DIR,
    check=True,
)

print("installed; Runtime -> Restart session, then run the cell below")

### ⚠️ Runtime → Restart session now

Then continue from the cell below. It re-derives every path it needs, so
nothing above has to run again — and it is also the first cell to re-run after
a disconnect.

In [ ]:
# 3. Session setup: Drive, cache paths, HF_HOME. Self-contained on purpose --
#    this is the cell to run first after a restart or a dropped connection.
import os
import sys
from pathlib import Path

# Before any HuggingFace import in this process. Set afterwards it does nothing:
# the cache location is fixed at import time, and the checkpoint would land on
# the ephemeral runtime disk to be re-downloaded every session.
assert not any(m.startswith(("huggingface_hub", "transformers")) for m in sys.modules), (
    "a HuggingFace module is already imported; Runtime -> Restart session and run this cell first"
)

try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
except ModuleNotFoundError:
    DRIVE_ROOT = Path("./drive_local")   # off Colab: keep the notebook runnable

# Storage is the Drive of whichever account is signed into Colab -- 15 GB on a
# personal account, Pro included. `drive.mount` cannot reach a second account, so
# moving to a larger Drive means running Colab from that account, not editing a
# path here. Full-corpus estimate: ~1.2 GB parcel + ~3.5 GB vertex + ~7 GB HF.
PROJECT_DRIVE = DRIVE_ROOT / "NeuroTutorSim"
CACHE_ROOT = PROJECT_DRIVE / "tribe_cache"
HF_CACHE = PROJECT_DRIVE / "hf"
HANDOFF = PROJECT_DRIVE / "gate17"          # written by notebook 01, read by 02
REPO_DIR = Path("/content/neurotutorsim")
for d in (PROJECT_DRIVE, CACHE_ROOT, HF_CACHE, HANDOFF):
    d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
# Exported, not just assigned: the `!` cells below launch a subprocess, which
# inherits the environment but knows nothing about this kernel's variables.
os.environ["CACHE_ROOT"] = str(CACHE_ROOT)

print(f"repo       : {REPO_DIR}  (exists={REPO_DIR.exists()})")
print(f"cache root : {CACHE_ROOT}")
print(f"HF cache   : {HF_CACHE}")

In [ ]:
# 4. Authenticate. HF_TOKEN comes from Colab Secrets (the key icon), never a literal.
#    `meta-llama/Llama-3.2-3B` (TRIBE's text encoder) is gated per account: a read
#    token grants nothing until Meta approves the request for that account. Colab
#    Secrets are per Google account too, so a new account needs HF_TOKEN added
#    again with notebook access toggled on.
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("authenticated")

## Re-pinning the revision — normally skipped

**The revision is already pinned and committed.** `config/tribe.yaml` names the
SHA and `config/checkpoints.lock` holds its checksums, so a fresh clone arrives
ready to verify. **Skip to *Run gate 17*.**

The three cells below exist for one situation: deliberately moving the pin to a
different checkpoint revision. Read the SHA off the Hub, write it into this
session's config, record the new checksums. `--write-lock` refuses to overwrite
an existing entry, so a re-pin is also a decision about the old one.

**The clone is ephemeral, so an edit here is not the pin of record.** The last
section copies the files to Drive; a new pin counts only once committed from a
machine with push access.

In [ ]:
# Print the checkpoint's current commit SHA. Nothing is written.
!cd /content/neurotutorsim && python scripts/run_tribe_verification.py \
    --config config/tribe.yaml --resolve-revision

In [ ]:
# Paste the SHA printed above, then run. This rewrites `checkpoint_revision` in
# this session's copy of config/tribe.yaml: one key, matched on its own line, so
# a malformed edit fails here instead of surfacing later as a wrong cache key.
import re
import sys
from pathlib import Path

REVISION_SHA = ""   # <- 40 lowercase hex characters, from the cell above

assert re.fullmatch(r"[0-9a-f]{40}", REVISION_SHA), (
    "REVISION_SHA must be the 40-character lowercase hex SHA printed above"
)

config_path = Path("/content/neurotutorsim/config/tribe.yaml")
patched, n = re.subn(
    r"(?m)^checkpoint_revision:.*$",
    f'checkpoint_revision: "{REVISION_SHA}"',
    config_path.read_text(encoding="utf-8"),
)
if n != 1:
    raise SystemExit(f"expected exactly one checkpoint_revision line, matched {n}")
config_path.write_text(patched, encoding="utf-8")

# Read it back through the real loader -- that is the check that the pin is valid.
sys.path.insert(0, "/content/neurotutorsim")
from src.tribe.config import load_config

print("pinned:", load_config(config_path).require_resolved_revision())

In [ ]:
# Record the checkpoint's file checksums for the revision just pinned.
# A separate, explicit step: a lock that writes itself verifies nothing. It also
# refuses to overwrite an existing entry -- a changed checksum for a pinned
# revision means either the pin or the download is wrong, and both need a human.
!cd /content/neurotutorsim && python scripts/run_tribe_verification.py \
    --config config/tribe.yaml --write-lock

## Run gate 17

Checksums the pinned checkpoint against the lock, reproduces Meta's published
example, asserts the output mesh is fsaverage5 (20484 vertices), records the
environment, and writes the gate artifact.

In [ ]:
!cd /content/neurotutorsim && python scripts/run_tribe_verification.py \
    --config config/tribe.yaml --cache-root "$CACHE_ROOT"

## Before closing the session

Four files are the evidence that gate 17 was cleared on this hardware, and all
four live in the ephemeral clone:

| file | why it matters | tracked in git? |
|---|---|---|
| `outputs/verification/gate17.json` | `run_tribe_inference.py` refuses to start without it; notebook 02 restores it from Drive | no — `outputs/` is gitignored, Drive is its home |
| `config/tribe.yaml` | holds the resolved revision | yes — already committed |
| `config/checkpoints.lock` | the checksums gate 17 verifies against | yes — already committed |
| `docs/tribe_environment.md` | §6.1 item 15 environment record | yes — commit it if this run produced a new one |

The cell below copies all four to `MyDrive/NeuroTutorSim/gate17/`.

Only `docs/tribe_environment.md` normally needs to travel back into git now
that the pin is committed — and only when the hardware or package versions
changed. If you re-pinned above, `config/tribe.yaml` and
`config/checkpoints.lock` must be committed too, **from a machine with push
access**, or the next session starts from an unpinned checkpoint.

In [ ]:
import shutil
from pathlib import Path

for relative in (
    "outputs/verification/gate17.json",
    "config/tribe.yaml",
    "config/checkpoints.lock",
    "docs/tribe_environment.md",
):
    source = REPO_DIR / relative
    if source.exists():
        shutil.copy2(source, HANDOFF / source.name)   # flat: notebook 02 reads by name
        print(f"copied  {relative}  ->  {HANDOFF / source.name}")
    else:
        print(f"MISSING {relative}  (gate 17 did not get that far)")

environment = REPO_DIR / "docs" / "tribe_environment.md"
if environment.exists():
    print()
    print(environment.read_text(encoding="utf-8")[:800])